In [1]:
import pickle
import pandas as pd
import mlflow
import os


from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import make_pipeline
from fastapi import params

In [2]:
RUN_ID = os.getenv('RUN_ID', '1a34c4da71f2449da99ad5c6d7bf9d1f')

logged_model = f"s3://mlflow-model-avad/1/{RUN_ID}/artifacts/model"
model = mlflow.pyfunc.load_model(logged_model)

# mlflow.set_tracking_uri("http://127.0.0.1:5000")
# mlflow.set_experiment("green-taxi-ride-duration")

MlflowException: The following failures occurred while downloading one or more artifacts from s3://mlflow-model-avad/1/1a34c4da71f2449da99ad5c6d7bf9d1f/artifacts:
##### File model #####
An error occurred (404) when calling the HeadObject operation: Not Found

In [4]:
def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds()
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df

def prepare_dictionaries(df: pd.DataFrame):
    df["PU_DO"] = df["PULocationID"] + "_" + df["DOLocationID"]
    categorical = ["PU_DO"]
    numerical = ["trip_distance"]
    dicts = df[categorical + numerical].to_dict(orient="records")
    return dicts


In [5]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

dict_train = prepare_dictionaries(df_train)
dict_val = prepare_dictionaries(df_val)

In [6]:
with mlflow.start_run():
    params = dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=0)
    mlflow.log_params(params)

    pipeline = make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs=-1)
    )

    pipeline.fit(dict_train, y_train)
    y_pred = pipeline.predict(dict_val)

    rmse = root_mean_squared_error(y_val, y_pred)
    print(params, rmse)
    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(pipeline, artifact_path="model")

2026/04/13 11:06:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 11:06:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 16.53467808612593
🏃 View run judicious-lamb-960 at: http://127.0.0.1:5000/#/experiments/1/runs/1a34c4da71f2449da99ad5c6d7bf9d1f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [9]:
from mlflow.tracking import MlflowClient

In [10]:
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
RUN_ID = "09eabed7f5a6478eb99aeb00e8e01904"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [13]:
path = client.download_artifacts(run_id=RUN_ID, path="dict-vectorizer.bin")

In [14]:
with open(path, "rb") as f_out:
    dv = pickle.load(f_out)

In [15]:
dv

,"dtype dtype: dtype, default=np.float64The type of feature values. Passed to Numpy array/scipy.sparse matrixconstructors as the dtype argument.",<class 'numpy.float64'>
,"separator separator: str, default=""=""Separator string used when constructing new features for one-hotcoding.",'='
,"sparse sparse: bool, default=TrueWhether transform should produce scipy.sparse matrices.",True
,"sort sort: bool, default=TrueWhether ``feature_names_`` and ``vocabulary_`` should besorted when fitting.",True
